In [2]:
import geopandas as gpd
import folium
import osmnx as ox
import os

In [3]:
# # street network 

# G = ox.graph_from_place(
#   'Christchurch, New Zealand',
#   network_type='drive',
#   simplify=True,
# )

# G = ox.project_graph(G, to_crs=2193)

# edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
# nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)

In [4]:
property = gpd.read_file('output/property.gpkg', engine='pyogrio')

access_points = gpd.read_file('output/property_accesspoints.gpkg', engine='pyogrio')

In [5]:
property_geom = property[['property_id', 'geometry']].copy()

property_geom = gpd.GeoDataFrame(
  property_geom,
  crs=property.crs,
  geometry="geometry"
)

In [6]:
access_points = access_points.rename(columns={'geometry': 'access_point'})

property_reaches = property_geom.merge(
    access_points[['property_id', 'access_point']],    
    on='property_id',
    how='left'
)

In [7]:
REACHES = [50, 100, 150, 200, 250, 300, 350, 400]

for distance in REACHES:
  path = f'output/property_reach_{distance}m.gpkg'
  reach_col = f'reach_{distance}m'
  
  if os.path.exists(path):
      reach_gdf = gpd.read_file(path, engine='pyogrio')
      reach_gdf = reach_gdf.rename(columns={'geometry': reach_col})
      
      property_reaches = property_reaches.merge(
          reach_gdf[['property_id', reach_col]],    
          on='property_id',
          how='left'
      )

In [8]:
sample_size=10

In [9]:
sub_property = property_reaches.sample(n=sample_size, random_state=10).copy()

property_wgs = sub_property.to_crs(epsg=4326)

property_wgs["access_point"] = gpd.GeoSeries(
    sub_property["access_point"],
    crs=property_reaches.crs
).to_crs(epsg=4326)

for distance in REACHES:
  reach_col = f'reach_{distance}m'
  
  property_wgs[reach_col] = gpd.GeoSeries(
      sub_property[reach_col],
      crs=property_reaches.crs
  ).to_crs(epsg=4326)

In [10]:
m = folium.Map(
  location=[property_wgs.geometry.y.mean(), property_wgs.geometry.x.mean()],
  zoom_start=14,
  tiles='CartoDB positron'
)

# edges_wgs = edges.to_crs(epsg=4326)
# nodes_wgs = nodes.to_crs(epsg=4326)

# folium.GeoJson(
#     edges_wgs,
#     name="Street Network",
#     style_function=lambda x: {
#         "color": "gray",
#         "weight": 1,
#         "opacity": 0.5
#     }
# ).add_to(m)

# folium.GeoJson(
#     nodes_wgs,
#     name="Street Nodes",
#     marker=folium.CircleMarker(
#         radius=1,
#         color="black",
#         fill=True,
#         fill_opacity=0.7
#     )
# ).add_to(m)

REACHES = [
    "reach_400m",
    "reach_350m",
    "reach_300m",
    "reach_200m",
    "reach_250m",
    "reach_150m",
    "reach_100m",
    "reach_50m",
]

REACH_COLORS = {
    "reach_50m":  "#2c7bb6",
    "reach_100m": "#00a6ca",
    "reach_150m": "#00ccbc",
    "reach_200m": "#90eb9d",
    "reach_250m": "#ffff8c",
    "reach_300m": "#f9d057",
    "reach_350m": "#f29e2e",
    "reach_400m": "#d7191c",
}

reach_layers = {
    reach: folium.FeatureGroup(name=reach)
    for reach in REACHES
}

for idx, row in property_wgs.iterrows():
  # Property point
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x],
        radius=6,
        color="purple",
        fill=True,
        fill_opacity=1,
        popup=f"Property ID: {idx}"
    ).add_to(m)

    # Access point
    if row["access_point"] is not None:
        folium.CircleMarker(
            [row["access_point"].y, row["access_point"].x],
            radius=4,
            color="blue",
            fill=True,
            fill_opacity=0.9,
            popup="Access point"
        ).add_to(m)

    for reach in REACHES:
        geom = row.get(reach)

        if geom is None or geom.is_empty:
            continue

        folium.GeoJson(
            geom,
            name=reach,
            style_function=lambda x, c=REACH_COLORS[reach]: {
                "color": c,
                "weight": 3,
                "opacity": 0.6
            },
            tooltip=f"{reach.replace('_', ' ').upper()}"
        ).add_to(reach_layers[reach])
        
for layer in reach_layers.values():
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
        
m